In [23]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import precision_recall_fscore_support

In [24]:
dataset = load_dataset("coastalcph/tydi_xor_rc")


df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()



#filter data 
langlst = ['ko','ar','te']
# Source - https://stackoverflow.com/a/59275490
# Posted by atinjanki
# Retrieved 2026-09-05, License - CC BY-SA 4.0

df_train_filtered = df_train[df_train['lang'].isin(langlst)]
df_validation_filtered = df_validation[df_validation['lang'].isin(langlst)]

#filter data 
#Training data for each language
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_te = df_train[(df_train['lang'] == "te")]

#Validation data for each language
df_validation_ar = df_validation[(df_validation['lang'] == "ar")]
df_validation_ko = df_validation[(df_validation['lang'] == "ko")]
df_validation_te = df_validation[(df_validation['lang'] == "te")]



df_train_filtered


,question,context,lang,answerable,answer_start,answer,answer_inlang
4792,30년 전쟁의 승자는 누구인가?,"The conflict between France and Spain continued in Catalonia until 1659, with the confrontation between two sovereigns and two Catalan governments, one based in Barcelona, under the control of Spain and the other in Perpinyà, under the occupation of France. In 1652 the French authorities renounc...",ko,True,21,France,None
4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electromagnetic radiation. Most X-rays have a wavelength ranging from 0.01 to 10 nanometers, corresponding to frequencies in the range 30 petahertz to 30 exahertz (3×10<sup>16</sup> Hz to 3×10<sup>19</sup> Hz) and energies in the range 100 eV to 100 keV. X-r...",ko,True,503,Wilhelm Röntgen,None
4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever city that has held both the summer and the winter Olympic Games. Ten cities will have hosted the Olympic Games more than once: Athens (1896 and 2004 Summer Olympics), Paris (1900, 1924 and 2024 Summer Olympics), London (1908, 1948 and 2012 Summer Olymp...",ko,True,188,2004,None
4795,세상에서 가장 오래된 방송사는 무엇인가?,"The British Broadcasting Corporation (BBC) is a British public service broadcaster. Its headquarters are at Broadcasting House in Westminster, London, and it is the world's oldest national broadcasting organisation and the largest broadcaster in the world by number of employees. It employs over ...",ko,True,4,British Broadcasting Corporation (BBC),None
4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palestine ( '), is a ""de jure"" sovereign state in Western Asia claiming the West Bank (bordering Israel and Jordan) and Gaza Strip (bordering Israel and Egypt) with Jerusalem as the designated capital, although its administrative center is currently locate...",ko,True,205,Jerusalem,None
...,...,...,...,...,...,...,...
15338,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials met in the northeastern town of Garowe to discuss post-transition arrangements. After extensive deliberations attended by regional actors and international observers, the conference ended in a signed agreement between TFG President Sharif Sheikh Ahme...",ko,True,923,23 June 2012,None
15339,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,"The first earth tracks were created by humans carrying goods and often followed trails. Tracks would be naturally created at points of high traffic density. As animals were domesticated, horses, oxen and donkeys became an element in track-creation. With the growth of trade, tracks were often fla...",ko,True,160,animals,None
15340,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """" : born 19 November 1954) is an Egyptian politician who is the sixth and current President of Egypt, in office since 2014.",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,None
15341,독일에서 가장 인구밀도가 높은 도시는 무엇인가?,"Munich (; ; ) is the capital and most populous city of the second most populous German federal state of Bavaria, and, with a population of around 1.5 million, it is the third-largest city of Germany after Berlin and Hamburg, as well as the 12th-largest city in the European Union. The city's metr...",ko,True,205,Berlin,None


## Logistic-regression baseline

In [25]:
train_texts = df_train_filtered["question"] + " " + df_train_filtered["context"]
validation_texts = df_validation_filtered["question"] + " " + df_validation_filtered["context"]

vectorizer = CountVectorizer(min_df=2)
X_train = vectorizer.fit_transform(train_texts)
X_validation = vectorizer.transform(validation_texts)
y_train = df_train_filtered["answerable"]
y_validation = df_validation_filtered["answerable"]

logistic_baseline = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
)
logistic_baseline.fit(X_train, y_train)
validation_predictions = logistic_baseline.predict(X_validation)

In [26]:
results = []

for language in ["all", "ar", "ko", "te"]:
    if language == "all":
        true_labels = y_validation
        predicted_labels = validation_predictions
    else:
        language_rows = df_validation_filtered["lang"] == language
        true_labels = y_validation[language_rows]
        predicted_labels = validation_predictions[language_rows]

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predicted_labels, labels=[False, True]
    )
    results.append([
        language,
        len(true_labels),
        precision[0], recall[0], f1[0],
        precision[1], recall[1], f1[1],
        (f1[0] + f1[1]) / 2,
    ])

results = pd.DataFrame(
    results,
    columns=[
        "language", "n",
        "unanswerable_P", "unanswerable_R", "unanswerable_F1",
        "answerable_P", "answerable_R", "answerable_F1", "macro_F1",
    ],
)
results.round(3)

,language,n,unanswerable_P,unanswerable_R,unanswerable_F1,answerable_P,answerable_R,answerable_F1,macro_F1
0,all,1155,0.800,0.341,0.479,0.900,0.986,0.941,0.710
1,ar,415,0.900,0.865,0.882,0.981,0.986,0.984,0.933
2,ko,356,0.000,0.000,0.000,0.946,0.997,0.971,0.486
3,te,384,0.579,0.118,0.196,0.775,0.973,0.863,0.530


## Recurrent classifier

In [27]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

train_tokens = tokenizer(
    df_train_filtered["question"].tolist(),
    df_train_filtered["context"].tolist(),
    padding="max_length", truncation="only_second", max_length=256, return_tensors="pt",
)
validation_tokens = tokenizer(
    df_validation_filtered["question"].tolist(),
    df_validation_filtered["context"].tolist(),
    padding="max_length", truncation="only_second", max_length=256, return_tensors="pt",
)

train_labels = torch.tensor(df_train_filtered["answerable"].astype(int).to_numpy())

train_data = TensorDataset(
    train_tokens["input_ids"], train_tokens["attention_mask"], train_labels
)
validation_data = TensorDataset(
    validation_tokens["input_ids"], validation_tokens["attention_mask"]
)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
validation_loader = DataLoader(validation_data, batch_size=32)

In [28]:
class RecurrentAnswerabilityClassifier(nn.Module):
    def __init__(self, vocabulary_size, padding_index, embedding_size=64, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_size, padding_idx=padding_index)
        self.lstm = nn.LSTM(embedding_size, hidden_size, batch_first=True)
        self.classifier = nn.Linear(hidden_size, 2)

    def forward(self, token_ids, attention_mask):
        embeddings = self.embedding(token_ids)
        outputs, _ = self.lstm(embeddings)
        last_tokens = (attention_mask.sum(dim=1) - 1).tolist()
        last_outputs = []
        for example_number in range(len(token_ids)):
            last_outputs.append(outputs[example_number, last_tokens[example_number]])
        return self.classifier(torch.stack(last_outputs))

In [29]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

recurrent_model = RecurrentAnswerabilityClassifier(
    len(tokenizer), tokenizer.pad_token_id
).to(device)

unanswerable_count = (train_labels == 0).sum().item()
answerable_count = (train_labels == 1).sum().item()
class_weights = torch.tensor([
    len(train_labels) / (2 * unanswerable_count),
    len(train_labels) / (2 * answerable_count),
], device=device)
loss_function = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(recurrent_model.parameters(), lr=0.001)

for epoch in range(5):
    recurrent_model.train()
    total_loss = 0

    for token_ids, attention_mask, labels in train_loader:
        token_ids = token_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = recurrent_model(token_ids, attention_mask)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    print(f"Epoch {epoch + 1}: loss = {total_loss / len(train_data):.4f}")

Epoch 1: loss = 0.6785
Epoch 2: loss = 0.6358
Epoch 3: loss = 0.5211
Epoch 4: loss = 0.3449
Epoch 5: loss = 0.1889


In [30]:
recurrent_model.eval()
recurrent_predictions = []

with torch.no_grad():
    for token_ids, attention_mask in validation_loader:
        logits = recurrent_model(token_ids.to(device), attention_mask.to(device))
        recurrent_predictions.extend(logits.argmax(dim=1).bool().cpu().tolist())

validation_results = df_validation_filtered.copy()
validation_results["recurrent_prediction"] = recurrent_predictions
recurrent_results = []

for language in ["all", "ar", "ko", "te"]:
    group = validation_results if language == "all" else validation_results[validation_results["lang"] == language]
    true_labels = group["answerable"]
    predicted_labels = group["recurrent_prediction"]

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predicted_labels, labels=[False, True]
    )
    recurrent_results.append([
        language,
        len(true_labels),
        precision[0], recall[0], f1[0],
        precision[1], recall[1], f1[1],
        (f1[0] + f1[1]) / 2,
    ])

recurrent_results = pd.DataFrame(
    recurrent_results,
    columns=[
        "language", "n",
        "unanswerable_P", "unanswerable_R", "unanswerable_F1",
        "answerable_P", "answerable_R", "answerable_F1", "macro_F1",
    ],
)
recurrent_results.round(3)

,language,n,unanswerable_P,unanswerable_R,unanswerable_F1,answerable_P,answerable_R,answerable_F1,macro_F1
0,all,1155,0.216,0.165,0.187,0.867,0.901,0.884,0.535
1,ar,415,0.150,0.115,0.130,0.877,0.906,0.892,0.511
2,ko,356,0.118,0.211,0.151,0.953,0.911,0.932,0.541
3,te,384,0.333,0.183,0.236,0.772,0.883,0.824,0.530


## Validation errors

In [31]:
errors_by_model = []

for model, predictions in [
    ("logistic regression", validation_predictions),
    ("recurrent classifier", recurrent_predictions),
]:
    errors = df_validation_filtered[["lang", "question", "context", "answerable"]].copy()
    errors["prediction"] = predictions
    errors = errors[errors["prediction"] != errors["answerable"]].copy()
    errors["model"] = model
    errors["error_type"] = "false positive"
    errors.loc[errors["answerable"], "error_type"] = "false negative"
    errors_by_model.append(errors)

all_errors = pd.concat(errors_by_model)

error_counts = all_errors.groupby(["model", "lang", "error_type"]).size()
error_counts

model                 lang  error_type    
logistic regression   ar    false negative     5
                            false positive     7
                      ko    false negative     1
                            false positive    19
                      te    false negative     8
                            false positive    82
recurrent classifier  ar    false negative    34
                            false positive    46
                      ko    false negative    30
                            false positive    15
                      te    false negative    34
                            false positive    76
dtype: int64

In [32]:
selected_errors = []

for model in ["logistic regression", "recurrent classifier"]:
    for language in ["ar", "ko", "te"]:
        for error_type in ["false positive", "false negative"]:
            matching_errors = all_errors[
                (all_errors["model"] == model)
                & (all_errors["lang"] == language)
                & (all_errors["error_type"] == error_type)
            ]
            if len(matching_errors) > 0:
                selected_errors.append(matching_errors.iloc[[0]])

selected_errors = pd.concat(selected_errors).reset_index()
selected_errors = selected_errors.rename(columns={"index": "validation_index"})
print(f"Selected {len(selected_errors)} errors")
pd.set_option("display.max_colwidth", 300)
selected_errors[[
    "validation_index", "model", "lang", "error_type",
    "question", "context", "answerable",
]]

Selected 12 errors


,validation_index,model,lang,error_type,question,context,answerable
0,1714,logistic regression,ar,false positive,هل يمكن حماية الأطفال من التعرض إلي التحرش الجنسي ؟,"In the United States growing awareness of child sexual abuse has sparked an increasing number of civil lawsuits for monetary damages stemming from such incidents. Increased awareness of child sexual abuse has encouraged more victims to come forward, whereas in the past victims often kept their a...",False
1,1478,logistic regression,ar,false negative,هل مصرهي دولة نامية؟,"In recent years, new terms have emerged to describe the largest developing countries such as BRIC that stands for Brazil, Russia, India, and China, along with ""BRICET"" (BRIC + Eastern Europe and Turkey), ""BRICS"" (BRIC + South Africa), ""BRICM"" (BRIC + Mexico), MINT (Mexico, Indonesia, Nigeria and...",True
2,364,logistic regression,ko,false positive,시차는 중력과 관련이 있는가?,"Gravitational time dilation was first described by Albert Einstein in 1907 as a consequence of special relativity in accelerated frames of reference. In general relativity, it is considered to be a difference in the passage of proper time at different positions as described by a metric tensor of...",False
3,1214,logistic regression,ko,false negative,디젤 자동차는 누가 개발했는가?,"The diesel car markets are the same ones who pioneered various developments (Mercedes-Benz, BMW, Peugeot/Citroën, Fiat, Alfa Romeo, Volkswagen Group) There were also small diesel engines produced in England by British Leyland and Perkins. For reasons of economy the petrol BMC ""B"" series engine w...",True
4,2911,logistic regression,te,false positive,మలేరియా వ్యాధి కి మందు కనిపెట్టిన శాస్త్రవేత్త ఎవరు?,"Malaria is a disease spread by mosquitoes. Malaria is transmitted when the parasite enters the blood of a person. Parasites depend on the humans they live with for their food. ""Charles Louis Alphonse Lavren"", a French defense physician, was awarded the Nobel Prize in 1907 for discovering how mal...",False
5,93,logistic regression,te,false negative,తెలుగు పంచాంగం ప్రకారం నూతన సంవత్సరం ఏ ఇంగ్లీష్ నెలలో ప్రారంభమవుతుంది?,The Telugu year is the calendar year for the Telugu speaking people of . Each Yuga (Era) has a cycle of 60 years. Each year of Ugadi year has a specific name in Panchangam (Astrnomical calendar) based on astrological influences and the name of the year would represent the character of that year....,True
6,1417,recurrent classifier,ar,false positive,هل زار ابن بطوطة اليمن؟,"After the ""hajj"" in either 1328 or 1330, he made his way to the port of Jeddah on the Red Sea coast. From there he followed the coast in a series of boats making slow progress against the prevailing south-easterly winds. Once in Yemen he visited Zabīd and later the highland town of Ta'izz, where...",False
7,1431,recurrent classifier,ar,false negative,لماذا يعتبر نظام التعليم الفلندي من افضل الانظمة التعليمية؟,"Education in Finland is an education system with fully subsidised meals served to full-time students. The present education system in Finland consists of daycare programmes (for babies and toddlers) and a one-year ""pre-school"" (or kindergarten for six-year-olds); a nine-year compulsory basic com...",True
8,364,recurrent classifier,ko,false positive,시차는 중력과 관련이 있는가?,"Gravitational time dilation was first described by Albert Einstein in 1907 as a consequence of special relativity in accelerated frames of reference. In general relativity, it is considered to be a difference in the passage of proper time at different positions as described by a metric tensor of...",False
9,378,recurrent classifier,ko,false negative,세상에서 가장 큰 미식축구 경기장은 어디인가?,"The Rungrado 1st of May Stadium, also known as the May Day Stadium, is a multi-purpose stadium in Rungra Island, Pyongyang, North Korea which was opened on 1 May 1989. Its first major event was the 13th World Festival of Youth and Students. It is the largest stadium in the world, with a total ca...",True
